<a href="https://colab.research.google.com/github/VilasiniRamamoorthy/Mini_Project1/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data clean-up and load to db**

In [ ]:
%pip install pymysql

In [ ]:
%pip install pandas sqlalchemy pymysql

In [10]:
#Creating records in db reading from csv

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Step 1: Load the CSV files
#movies.csv
movies_df = pd.read_csv("movies.csv")

#movie_keywords.csv
movie_keywords_df = pd.read_csv("movie_keywords.csv")

#movie_generes.csv
movie_generes_df = pd.read_csv('movie_genres.csv')

#genres.csv
genres_df = pd.read_csv("genres.csv")

#crew.csv
crew_df = pd.read_csv("crew.csv")

#cast.csv
cast_df = pd.read_csv("cast.csv")

# Step 2: Inspect and check the data
for name, df in [("movies", movies_df),("movie_keywords", movie_keywords_df),("movie_genres", movie_generes_df),
 ("genres", genres_df), ("crew", crew_df), ("cast", cast_df)]:
    print(f"\n--- {name.upper()} DATA ---")
    print(df.head())
    print("\nNull values:")
    print(df.isnull().sum())
    print("\nTotal null values:", df.isnull().sum().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("Unique rows:", df.drop_duplicates().shape[0])

# Step 3: Remove duplicate rows (if appropriate)
movies_df = movies_df.drop_duplicates()
movie_keywords_df = movie_keywords_df.drop_duplicates()
movie_generes_df = movie_generes_df.drop_duplicates()
genres_df = genres_df.drop_duplicates()
crew_df = crew_df.drop_duplicates()
cast_df = cast_df.drop_duplicates()

# Step 4: Remove null or zero from budget and revenue to be replaced with mean
# Replace zero budget values with the mean of non-zero budget values
budget_mean = movies_df.loc[movies_df["budget"] != 0, "budget"].mean()
movies_df.loc[movies_df["budget"] == 0, "budget"] = budget_mean

# Replace zero revenue values with the mean of non-zero revenue values
revenue_mean = movies_df.loc[movies_df["revenue"] != 0, "revenue"].mean()
movies_df.loc[movies_df["revenue"] == 0, "revenue"] = revenue_mean



# Step 5: Enter your TiDB connection details
host = "gateway01.ap-northeast-1.prod.aws.tidbcloud.com"
port = 4000
user = "3k5gspKEptkYxvs.root"
password = "uJVYvVceTGGGT00O"
database = "tmdb_proj_vilasini"

# Step 6: Create the database connection
connection_url = (f"mysql+pymysql://{user}:{password}"
    f"@{host}:{port}/{database}"
    "?ssl_verify_cert=true&ssl_verify_identity=true"
)
engine = create_engine(connection_url, pool_recycle=300)

# Step 7: Upload DataFrame to SQL
movies_df.to_sql(
    name="cleaned_movies",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded in cleaned_movies successfully!")

movie_keywords_df.to_sql(
    name="cleaned_movie_keywords",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded in cleaned_movie_keywords successfully!")

movie_generes_df.to_sql(
    name="cleaned_movie_generes",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded in cleaned_movie_genres successfully!")

genres_df.to_sql(
    name="cleaned_generes",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded in cleaned_generes successfully!")

crew_df.to_sql(
    name="cleaned_crew",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded in cleaned_crew successfully!")

cast_df.to_sql(
    name="cleaned_cast",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded in cleaned_cast successfully!")

# Step 6: Verify the uploaded records
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_movies")
    )
    print("Total rows uploaded in cleaned_movies:", result.scalar())

    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_movie_keywords")
    )
    print("Total rows uploaded in cleaned_movie_keywords:", result.scalar())

    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_movie_generes")
    )
    print("Total rows uploaded in cleaned_movie_generes:", result.scalar())

    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_generes")
    )
    print("Total rows uploaded in cleaned_generes:", result.scalar())

    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_crew")
    )
    print("Total rows uploaded in cleaned_crew:", result.scalar())

    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_cast")
    )
    print("Total rows uploaded in cleaned_cast:", result.scalar())


engine.dispose()



--- MOVIES DATA ---
   movie_id            title original_language release_date  runtime  \
0    157336     Interstellar                en   2014-11-05      169   
1     27205        Inception                en   2010-07-15      148   
2     24428     The Avengers                en   2012-04-25      143   
3       155  The Dark Knight                en   2008-07-16      152   
4     19995           Avatar                en   2009-12-16      162   

        budget       revenue  popularity  vote_average  vote_count    status  \
0  165000000.0  7.466067e+08     73.0452         8.484       40566  Released   
1  160000000.0  8.390306e+08     55.8497         8.400       39727  Released   
2  220000000.0  1.518816e+09     72.2515         8.062       38993  Released   
3  185000000.0  1.004558e+09     60.5510         8.534       36255  Released   
4  237000000.0  2.923706e+09     48.1996         7.610       34348  Released   

                                            overview  
0  The adv